In [1]:
import SimpleITK as sitk
import gui
import registration_gui as rgui
%matplotlib notebook
from pathlib import Path
import numpy as np
import os

In [13]:
fixed_image_f = Path("/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/reference_image/oriented_image.nii.gz")
moving_image_f = Path("/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir/wt_F_82Z-10879/ct_image_.nii.gz")

In [14]:
fixed_image =  sitk.ReadImage(fixed_image_f, sitk.sitkFloat32)
moving_image = sitk.ReadImage(moving_image_f, sitk.sitkFloat32)


In [17]:
def read_fcsv(file_name):
    points = Path(file_name).read_text().split("\n")[3:]
    
    points = [ x.split(',')[1:4] for x in points]
    
    res_points = []
    for x in points:
        res_points.append( [float(y) for y in x] )
    return res_points

fixed_image_points = read_fcsv((fixed_image_f.parent / "oriented.fcsv" ))
moving_image_points = read_fcsv( ( moving_image_f.parent / "label.fcsv" ) )

In [30]:
#fix movinf
fixed_image_points_flat = [c for p in fixed_image_points for c in p]        
moving_image_points_flat = [c for p in moving_image_points for c in p]
initial_transformation = sitk.LandmarkBasedTransformInitializer(sitk.VersorRigid3DTransform(), 
                                                                fixed_image_points_flat, 
                                                                moving_image_points_flat,referenceImage=fixed_image)

moving_resampled = sitk.Resample(moving_image, fixed_image, initial_transformation, sitk.sitkLinear, 0.0, moving_image.GetPixelID())

sitk.WriteImage(moving_resampled, str(moving_image_f.parent / "oriented_image1.nii.gz")  )


In [ ]:
ct_window_level = [835,162]
mr_window_level = [1036,520]
gui.RegistrationPointDataAquisition(fixed_image, moving_image, figure_size=(8,4), known_transformation=initial_transformation, fixed_window_level=ct_window_level, moving_window_level=mr_window_level);

In [ ]:
[ x.split(',')[1:4] for x in points]

In [31]:
initial_transformation.GetMatrix()

(-0.9388005181832377,
 0.3087227987567433,
 -0.15278684690341796,
 -0.33298958451671684,
 -0.9268931259869921,
 0.17316774988850187,
 -0.08815624573601902,
 0.21344640199793918,
 0.9729692234659145)

In [32]:
initial_transformation.GetTranslation()

(-23.60233333333333, -22.24433333333333, -8.902999999999997)